<a href="https://colab.research.google.com/github/Sarah-0405/Cold_Spots_Bayern/blob/main/knn_gistar_cold_spot_berechnung.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
!pip install geemap
!pip install geopandas
!pip install geopy
!pip install folium
!pip install matplotlib
!pip install numpy
!pip install pandas
!pip install rasterio
!pip install seaborn
!pip install shapely
!pip install sklearn
!pip install pysal

  Using cached sklearn-0.0.post12.tar.gz (2.6 kB)
  error: subprocess-exited-with-error
  
  × python setup.py egg_info did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  Preparing metadata (setup.py) ... error
error: metadata-generation-failed

× Encountered error while generating package metadata.
╰─> See above for output.

note: This is an issue with the package mentioned above, not pip.
hint: See above for details.


In [5]:
!pip install pysal

In [6]:
# Importieren Sie notwendige Bibliotheken
import ee
import geemap
import folium
from datetime import datetime
import geopandas as gpd
from shapely.geometry import mapping, Point
import pandas as pd
import matplotlib.pyplot as plt
import pysal.lib as ps
import pysal.explore as pe
from esda.getisord import G_Local
from libpysal.weights import KNN
from sklearn.preprocessing import StandardScaler
import numpy as np

/usr/local/lib/python3.11/dist-packages/spaghetti/network.py:41: FutureWarning: The next major release of pysal/spaghetti (2.0.0) will drop support for all ``libpysal.cg`` geometries. This change is a first step in refactoring ``spaghetti`` that is expected to result in dramatically reduced runtimes for network instantiation and operations. Users currently requiring network and point pattern input as ``libpysal.cg`` geometries should prepare for this simply by converting to ``shapely`` geometries.
  warnings.warn(dep_msg, FutureWarning, stacklevel=1)


# KNN-Matrix definieren


geodataframe laden

In [7]:
all_cities_summer_avg_lst_per_pixel = gpd.read_file("/content/drive/MyDrive/Cold Spots Bayern/all_cities_summer_avg_lst_per_pixel_allyears.geojson")
display(all_cities_summer_avg_lst_per_pixel.head())

,longitude,latitude,year,city,avg_summer_LST_Celsius,geometry
0,9.080523,50.007251,2019,Aschaffenburg,32.516656,POINT (9.08052 50.00725)
1,9.080524,50.007521,2019,Aschaffenburg,32.417533,POINT (9.08052 50.00752)
2,9.080524,50.007791,2019,Aschaffenburg,32.320120,POINT (9.08052 50.00779)
3,9.080941,50.006711,2019,Aschaffenburg,32.817441,POINT (9.08094 50.00671)
4,9.080941,50.006981,2019,Aschaffenburg,32.648249,POINT (9.08094 50.00698)


In [9]:
import os

In [10]:
# Annahme: Ihr zusammengeführter GeoDataFrame heißt 'all_cities_summer_avg_lst_per_pixel_allyears'
# Laden Sie den zusammengeführten GeoDataFrame, falls er noch nicht im Speicher ist
# Stellen Sie sicher, dass der Dateipfad korrekt ist
try:
    # Versuchen Sie zuerst, die neu erstellte Datei zu laden, falls vorhanden
    merged_geojson_path = all_cities_summer_avg_lst_per_pixel

    # merged_geojson_path = "/content/drive/MyDrive/Cold Spots Bayern/all_cities_summer_avg_lst_per_pixel_allyears.geojson"

    #all_cities_summer_avg_lst_per_pixel_year = gpd.read_file(merged_geojson_path)
    #print(f"Zusammengeführter GeoDataFrame geladen von: {merged_geojson_path}")
    #print(f"Anzahl der Pixel-Jahres-Einträge im geladenen GeoDataFrame: {len(all_cities_summer_avg_lst_per_pixel_year)}")

except Exception as e:
    print(f"Fehler beim Laden des zusammengeführten GeoDataFrames: {e}")
    print("Bitte stellen Sie sicher, dass der GeoDataFrame 'all_cities_summer_avg_lst_per_pixel_year' korrekt geladen ist.")
    # Beenden Sie hier, wenn der GeoDataFrame nicht geladen werden konnte
    # exit() # Oder eine andere geeignete Fehlerbehandlung

# Überprüfen Sie, ob der GeoDataFrame geladen wurde und nicht leer ist
if 'all_cities_summer_avg_lst_per_pixel' in locals() and not all_cities_summer_avg_lst_per_pixel.empty:

    # Holen Sie sich die Liste der eindeutigen Städte
    unique_cities = all_cities_summer_avg_lst_per_pixel['city'].unique()
    print(f"\nEindeutige Städte im GeoDataFrame: {list(unique_cities)}")

    # Definieren Sie den Ordner, in dem die stadtspezifischen GDFs gespeichert werden sollen
    # Sie können dies anpassen, z.B. einen temporären Ordner im Colab-Dateisystem verwenden
    output_folder = "/content/temp_city_gdfs/"
    os.makedirs(output_folder, exist_ok=True) # Erstellt den Ordner, falls er nicht existiert

    print(f"\nFiltere Daten pro Stadt und speichere in {output_folder}...")

    # Dictionary zum Speichern der stadtspezifischen GeoDataFrames (optional, falls Sie sie später im Speicher benötigen)
    city_gdfs = {}

    # Schleife über jede Stadt
    for city in unique_cities:
        print(f"  Verarbeite Stadt: {city}...")

        # Filtern Sie die Daten für die aktuelle Stadt (alle Jahre für diese Stadt)
        gdf_city = all_cities_summer_avg_lst_per_pixel[
            all_cities_summer_avg_lst_per_pixel['city'] == city
        ].copy() # Wichtig: Kopie erstellen

        if not gdf_city.empty:
            # Optional: Speichern Sie den gefilterten GeoDataFrame lokal
            # Verwenden Sie einen Dateinamen, der den Stadtnamen enthält
            output_file_name = f"{city.replace(' ', '_')}_avg_summer_lst_per_pixel_allyears.geojson"
            output_file_path = os.path.join(output_folder, output_file_name)

            try:
                # Speichern Sie den GeoDataFrame
                gdf_city.to_file(output_file_path, driver='GeoJSON')
                print(f"    Gespeichert: {output_file_path} ({len(gdf_city)} Pixel-Jahres-Einträge)")

                # Optional: Speichern Sie den GeoDataFrame auch im Dictionary im Speicher
                city_gdfs[city] = gdf_city

            except Exception as e:
                print(f"    Fehler beim Speichern des GeoDataFrames für {city}: {e}")

        else:
            print(f"    Keine Daten für Stadt {city} gefunden.")

    print("\nFertig mit dem Filtern und Speichern der stadtspezifischen GeoDataFrames.")

    # Jetzt haben Sie einzelne GeoJSON-Dateien für jede Stadt im angegebenen Ausgabeordner.
    # Sie haben auch ein Dictionary 'city_gdfs', das die GeoDataFrames im Speicher enthält (optional).

else:
    print("Der zusammengeführte GeoDataFrame ('all_cities_summer_avg_lst_per_pixel_year') ist nicht verfügbar oder leer. Filtern nicht möglich.")


Eindeutige Städte im GeoDataFrame: ['Aschaffenburg', 'Augsburg', 'Bamberg', 'Bayreuth', 'Erlangen', 'Fürth', 'Ingolstadt', 'Kempten (Allgäu)', 'Landshut', 'Munich', 'Nuremberg', 'Passau', 'Regensburg', 'Rosenheim', 'Schweinfurt', 'Würzburg']

Filtere Daten pro Stadt und speichere in /content/temp_city_gdfs/...
  Verarbeite Stadt: Aschaffenburg...
    Gespeichert: /content/temp_city_gdfs/Aschaffenburg_avg_summer_lst_per_pixel_allyears.geojson (362052 Pixel-Jahres-Einträge)
  Verarbeite Stadt: Augsburg...


KeyboardInterrupt: 

hier wird für KNN Matrix der zusammengeführte gdf aller Städte verwendet

In [ ]:
# Annahme: Ihr zusammengeführter GeoDataFrame heißt 'all_cities_avg_summer_lst_per_pixel_year'
# Laden Sie den zusammengeführten GeoDataFrame, falls er noch nicht im Speicher ist
# Stellen Sie sicher, dass der Dateipfad korrekt ist
try:
    # Versuchen Sie zuerst, die neu erstellte Datei zu laden, falls vorhanden
    merged_geojson_path = all_cities_summer_avg_lst_per_pixel
    # oder, wenn noch nicht vorher geladen: merged_geojson_path = "/content/drive/MyDrive/Cold Spots Bayern/all_cities_summer_avg_lst_per_pixel_allyears.geojson"

    # all_cities_avg_summer_lst_per_pixel_year = gpd.read_file(merged_geojson_path)
    # print(f"Zusammengeführter GeoDataFrame geladen von: {merged_geojson_path}")
    # print(f"Anzahl der Pixel-Jahres-Einträge im geladenen GeoDataFrame: {len(all_cities_avg_summer_lst_per_pixel_year)}")

    # Stellen Sie sicher, dass die Geometrie vorhanden ist und gültig ist
    if 'geometry' not in all_cities_summer_avg_lst_per_pixel.columns or all_cities_summer_avg_lst_per_pixel.geometry.isnull().any():
        print("⚠️ Warnung: Geometrie-Spalte fehlt oder enthält Null-Werte. KNN-Berechnung nicht möglich.")
        # Hier könnten Sie eine Fehlerbehandlung oder ein Überspringen implementieren
    else:
        print("Geometrie-Spalte ist vorhanden und gültig.")


except Exception as e:
    print(f"Fehler beim Laden des zusammengeführten GeoDataFrames: {e}")
    print("Bitte stellen Sie sicher, dass der GeoDataFrame 'all_cities_summer_avg_lst_per_pixel' korrekt geladen ist.")
    # Beenden Sie hier, wenn der GeoDataFrame nicht geladen werden konnte
    # exit() # Oder eine andere geeignete Fehlerbehandlung


# Überprüfen Sie, ob der GeoDataFrame geladen wurde und nicht leer ist
if 'all_cities_summer_avg_lst_per_pixel' in locals() and not all_cities_summer_avg_lst_per_pixel.empty:

    # Holen Sie sich die Liste der eindeutigen Städte
    unique_cities = all_cities_summer_avg_lst_per_pixel['city'].unique()
    print(f"\nEindeutige Städte im GeoDataFrame: {list(unique_cities)}")

    # Dictionary zum Speichern der KNN-Gewichtsmatrizen für jede Stadt
    city_knn_weights = {}

    # Definieren Sie die Anzahl der Nachbarn für KNN
    k_neighbors = 12

    print(f"\nBerechne KNN-Gewichtsmatrizen (k={k_neighbors}) für jede Stadt einzeln...")

    # Schleife über jede Stadt
    for city in unique_cities:
        print(f"\n✨ Verarbeite Stadt: {city}...")

        # Dictionary zum Speichern der KNN-Gewichtsmatrizen pro Jahr für die aktuelle Stadt
        city_yearly_knn_weights = {}

        # Holen Sie sich die eindeutigen Jahre für diese Stadt
        years_in_city = sorted(all_cities_summer_avg_lst_per_pixel[
            all_cities_summer_avg_lst_per_pixel['city'] == city
        ]['year'].unique())

        if not years_in_city:
            print(f"  Keine Jahresdaten für Stadt {city} gefunden. Überspringe.")
            continue


        for year in years_in_city:
             print(f"  Verarbeite Jahr: {year} für Stadt {city}...")

             # Filtern Sie die Daten für die aktuelle Stadt und das aktuelle Jahr
             gdf_city_year = all_cities_summer_avg_lst_per_pixel[
                 (all_cities_summer_avg_lst_per_pixel['city'] == city) &
                 (all_cities_summer_avg_lst_per_pixel['year'] == year)
             ].copy() # Wichtig: Kopie erstellen, um SettingWithCopyWarning zu vermeiden

             # Stellen Sie sicher, dass genügend Punkte für die KNN-Analyse vorhanden sind
             # KNN benötigt mindestens k+1 Punkte.
             min_points_for_knn = k_neighbors + 1
             if not gdf_city_year.empty and len(gdf_city_year) >= min_points_for_knn:
                 # Extrahieren Sie die Koordinaten
                 # Stellen Sie sicher, dass das CRS projiziert ist oder konvertieren Sie es kurzfristig
                 # Konvertiere in ein metrisches CRS vor der KNN-Berechnung, wenn es noch nicht projiziert ist
                 gdf_city_year_spatial = gdf_city_year.copy() # Kopie erstellen
                 if gdf_city_year_spatial.crs is None or gdf_city_year_spatial.crs.is_geographic:
                     # Versuche, ein geeignetes UTM-CRS basierend auf der Stadt zu finden, oder verwende ein Standard-CRS
                     # Für Bayern ist EPSG:25832 (ETRS89 / UTM zone 32N) oft passend.
                     try:
                         print(f"    Konvertiere Daten für {city} im Jahr {year} nach EPSG:25832 (UTM 32N) für metrische Distanzen.")
                         gdf_city_year_spatial = gdf_city_year_spatial.to_crs(epsg=25832)
                     except Exception as crs_e:
                         print(f"    Fehler bei der CRS-Konvertierung für {city} im Jahr {year}: {crs_e}. Versuche es ohne Konvertierung, Ergebnisse könnten ungenau sein.")
                         # Wenn die Konvertierung fehlschlägt, verwenden Sie die Originalkoordinaten, aber mit Warnung
                         coords = np.array(list(zip(gdf_city_year.geometry.x, gdf_city_year.geometry.y)))
                         # Passen Sie hier auf, da die Distanzberechnung dann auf Grad basiert, was nicht ideal ist.
                         # Sie könnten auch eine Distanzmetrik angeben, die mit geografischen Koordinaten umgehen kann (z.B. Balltree oder KDTree mit metrischer Distanzberechnung, aber das erfordert oft projizierte Daten).
                         # Für Einfachheit und da es oft funktioniert, lassen wir libpysal versuchen, die Distanzen zu berechnen.
                         # Sei dir aber bewusst, dass die Distanzen in diesem Fall in Grad sind, wenn das CRS geografisch ist.
                         pass # Geht weiter mit den ursprünglichen Koordinaten

                 coords = np.array(list(zip(gdf_city_year_spatial.geometry.x, gdf_city_year_spatial.geometry.y)))


                 # Berechne die KNN-Matrix
                 try:
                     w_city_year = KNN.from_array(coords, k=k_neighbors)
                     w_city_year.transform = 'R' # Zeilenstandardisierung anwenden

                     # Prüfen auf Konnektivität
                     if w_city_year.n_components > 1:
                          print(f"    Warnung: Gewichtsmatrix für {city} im Jahr {year} ist nicht vollständig verbunden ({w_city_year.n_components} Komponenten).")

                     # Speichere die Gewichtsmatrix im Dictionary
                     city_yearly_knn_weights[year] = w_city_year
                     print(f"    KNN-Matrix für {city} im Jahr {year} erfolgreich berechnet.")

                 except Exception as knn_e:
                     print(f"    Fehler bei der KNN-Berechnung für {city} im Jahr {year}: {knn_e}. Überspringe dieses Jahr.")
                     # Füge None hinzu, um anzuzeigen, dass für dieses Jahr keine Matrix erstellt wurde
                     city_yearly_knn_weights[year] = None


             else:
                 print(f"  Nicht genügend Daten ({len(gdf_city_year)} Punkte) für KNN-Berechnung für {city} im Jahr {year}. Benötigt mindestens {min_points_for_knn} Punkte. Überspringe.")
                 city_yearly_knn_weights[year] = None # Füge None hinzu, um anzuzeigen, dass keine Matrix erstellt wurde


        # Speichere das Dictionary der jährlichen Gewichtsmatrizen für die Stadt
        if city_yearly_knn_weights: # Nur speichern, wenn mindestens eine Matrix erstellt wurde
             city_knn_weights[city] = city_yearly_knn_weights
        else:
             print(f"Keine KNN-Matrizen für Stadt {city} in irgendeinem Jahr erstellt.")


    print("\nKNN-Berechnung für alle Städte abgeschlossen.")

    # Nun haben Sie ein Dictionary, 'city_knn_weights', das die KNN-Matrizen
    # für jede Stadt und jedes Jahr enthält.
    # Beispiel: city_knn_weights['Munich'][2019] gibt die KNN-Matrix für München im Jahr 2019 zurück.

    # Sie können nun mit diesen Matrizen die Gi* Analyse für jede Stadt und jedes Jahr durchführen.

else:
    print("Der zusammengeführte GeoDataFrame ('all_cities_avg_summer_lst_per_pixel_year') ist nicht verfügbar oder leer. KNN-Berechnung nicht möglich.")

jetzt wurde als Datensatz die lokal gespeicherten gdfs verwendet (für jede Stadt einzeln gespeicherte gdfs)

In [ ]:
# Definieren Sie den Ordner, in dem die stadtspezifischen GDFs gespeichert wurden
input_folder = "/content/temp_city_gdfs/"

# Holen Sie sich eine Liste aller GeoJSON-Dateien im temporären Ordner
# Diese Dateien sollten die aggregierten Daten pro Pixel, Jahr und Stadt enthalten
city_aggregated_geojson_files = [f for f in os.listdir(input_folder) if f.endswith("_avg_summer_lst_per_pixel_allyears.geojson")]

# Dictionary zum Speichern der KNN-Gewichtsmatrizen für jede Stadt und jedes Jahr
# Struktur: {'Stadtname': {Jahr: KNN_Matrix, ...}, ...}
city_yearly_knn_weights = {}

# Definieren Sie die Anzahl der Nachbarn für KNN
k_neighbors = 12

print(f"\nBerechne KNN-Gewichtsmatrizen (k={k_neighbors}) für jede Stadt und jedes Jahr einzeln aus temporären Dateien...")

# Schleife über jede Stadt-Datei im Ordner
for file_name in city_aggregated_geojson_files:
    file_path = os.path.join(input_folder, file_name)
    # Extrahiere den Stadtnamen aus dem Dateinamen
    city_name = file_name.replace("_avg_summer_lst_per_pixel_allyears.geojson", "").replace("_", " ")

    try:
        print(f"\n✨ Verarbeite Stadt: {city_name} (Datei: {file_name})...")

        # Lade den aggregierten GeoDataFrame für die aktuelle Stadt
        gdf_city_aggregated = gpd.read_file(file_path)
        print(f"  Geladen: {len(gdf_city_aggregated)} Pixel-Jahres-Einträge für {city_name}.")

        if not gdf_city_aggregated.empty:
            # Holen Sie sich die eindeutigen Jahre für diese Stadt aus dem geladenen DataFrame
            years_in_city = sorted(gdf_city_aggregated['year'].unique())
            print(f"  Verfügbare Jahre für {city_name}: {years_in_city}")

            if not years_in_city:
                print(f"  Keine Jahresdaten im aggregierten GDF für Stadt {city_name} gefunden. Überspringe.")
                continue

            # Dictionary zum Speichern der jährlichen Gewichtsmatrizen für die aktuelle Stadt
            current_city_yearly_weights = {}

            # Schleife über jedes Jahr für die aktuelle Stadt
            for year in years_in_city:
                 print(f"  Verarbeite Jahr: {year} für Stadt {city_name}...")

                 # Filtern Sie die Daten für das aktuelle Jahr innerhalb der aktuellen Stadt
                 # Der gdf_city_aggregated enthält bereits die durchschnittlichen LST-Werte pro Pixel pro Jahr
                 gdf_city_year_avg = gdf_city_aggregated[
                     gdf_city_aggregated['year'] == year
                 ].copy() # Wichtig: Kopie erstellen

                 # Stellen Sie sicher, dass genügend Punkte für die KNN-Analyse vorhanden sind
                 min_points_for_knn = k_neighbors + 1
                 if not gdf_city_year_avg.empty and len(gdf_city_year_avg) >= min_points_for_knn:
                     # Extrahieren Sie die Koordinaten
                     # Stellen Sie sicher, dass das CRS projiziert ist oder konvertieren Sie es kurzfristig
                     # Konvertiere in ein metrisches CRS vor der KNN-Berechnung, wenn es noch nicht projiziert ist
                     gdf_year_spatial = gdf_city_year_avg.copy() # Kopie erstellen
                     if gdf_year_spatial.crs is None or gdf_year_spatial.crs.is_geographic:
                         # Versuche, ein geeignetes UTM-CRS basierend auf der Stadt zu finden
                         # Für Bayern ist EPSG:25832 (ETRS89 / UTM zone 32N) oft passend.
                         try:
                             # print(f"    Konvertiere Daten für {city_name} im Jahr {year} nach EPSG:25832 (UTM 32N).")
                             gdf_year_spatial = gdf_year_spatial.to_crs(epsg=25832)
                         except Exception as crs_e:
                             print(f"    Fehler bei der CRS-Konvertierung für {city_name} im Jahr {year}: {crs_e}. Verwende Originalkoordinaten.")
                             # Wenn die Konvertierung fehlschlägt, verwenden Sie die Originalkoordinaten
                             # Beachten Sie die Warnung von libpysal bei geografischen Koordinaten
                             pass # Geht weiter mit den ursprünglichen Koordinaten


                     # Extrahieren Sie die Koordinaten aus der (potenziell konvertierten) Geometrie
                     coords = np.array(list(zip(gdf_year_spatial.geometry.x, gdf_year_spatial.geometry.y)))


                     # Berechne die KNN-Matrix
                     try:
                         w_city_year = KNN.from_array(coords, k=k_neighbors)
                         w_city_year.transform = 'R' # Zeilenstandardisierung anwenden

                         # Prüfen auf Konnektivität
                         if w_city_year.n_components > 1:
                              print(f"    Warnung: Gewichtsmatrix für {city_name} im Jahr {year} ist nicht vollständig verbunden ({w_city_year.n_components} Komponenten).")

                         # Speichere die Gewichtsmatrix im Dictionary
                         current_city_yearly_weights[year] = w_city_year
                         # print(f"    KNN-Matrix für {city_name} im Jahr {year} erfolgreich berechnet.") # Weniger Logging

                     except Exception as knn_e:
                         print(f"    Fehler bei der KNN-Berechnung für {city_name} im Jahr {year}: {knn_e}. Überspringe dieses Jahr.")
                         # Füge None hinzu, um anzuzeigen, dass für dieses Jahr keine Matrix erstellt wurde
                         current_city_yearly_weights[year] = None


                 else:
                     print(f"  Nicht genügend Daten ({len(gdf_city_year_avg)} Punkte) für KNN-Berechnung für {city_name} im Jahr {year}. Benötigt mindestens {min_points_for_knn} Punkte. Überspringe.")
                     current_city_yearly_weights[year] = None # Füge None hinzu


            # Speichere das Dictionary der jährlichen Gewichtsmatrizen für die Stadt, wenn Matrizen erstellt wurden
            if current_city_yearly_weights:
                 city_yearly_knn_weights[city_name] = current_city_yearly_weights
            else:
                 print(f"  Keine KNN-Matrizen für Stadt {city_name} in irgendeinem Jahr erstellt.")

        else:
             print(f"  GeoDataFrame für Stadt {city_name} war leer. Überspringe Verarbeitung.")

    except Exception as e:
        print(f"Fehler beim Laden oder Verarbeiten der Datei {file_path}: {e}. Überspringe diese Datei.")


print("\nKNN-Berechnung für alle Städte und Jahre abgeschlossen.")

# Das Dictionary 'city_yearly_knn_weights' enthält nun die KNN-Matrizen
# für jede Stadt und jedes Jahr, basierend auf den lokalen Dateien.
# Beispiel: city_yearly_knn_weights['Munich'][2020] gibt die KNN-Matrix für München im Jahr 2020 zurück.

in beiden Fällen: Laufzeit stürzt immer ab weil zu wenig RAM verfügbar

**KNN für Durchschnitt der Jahre 2019-2025** erstellen => erstmal gdfs Mittelwert der Jahre, dann KNN

In [11]:
# 1. Gruppierung und Mittelwertbildung
# Gruppieren nach 'city' und 'geometry' und berechnen des Mittelwerts für 'avg_summer_LST_Celsius'
# Setzen Sie 'geometry' als Index, um die Gruppierung zu vereinfachen und dann zurückzusetzen
averaged_lst_per_pixel_per_city = all_cities_summer_avg_lst_per_pixel.set_index('geometry').groupby(['city', 'geometry'])['avg_summer_LST_Celsius'].mean().reset_index()

# Konvertieren Sie das Ergebnis zurück in ein GeoDataFrame
# Annahme: Das ursprüngliche CRS des GeoDataFrames ist bekannt oder kann von der ersten Zeile abgeleitet werden.
# Wenn das ursprüngliche CRS nicht bekannt ist, müssen Sie es hier explizit festlegen.
# Beispiel: original_crs = all_cities_summer_avg_lst_per_pixel.crs
# Wenn das CRS None ist und Sie wissen, dass es sich um WGS84 (EPSG:4326) handelt:
# original_crs = "EPSG:4326"

# Verwenden Sie das CRS des ursprünglichen GeoDataFrames
original_crs = all_cities_summer_avg_lst_per_pixel.crs

averaged_lst_per_pixel_per_city = gpd.GeoDataFrame(
    averaged_lst_per_pixel_per_city,
    geometry='geometry',
    crs=original_crs # Setzen Sie das CRS des ursprünglichen GeoDataFrames
)

print("Mittelwert der LST pro Pixel pro Stadt über alle Jahre berechnet.")
display(averaged_lst_per_pixel_per_city.head())

Mittelwert der LST pro Pixel pro Stadt über alle Jahre berechnet.


,city,geometry,avg_summer_LST_Celsius
0,Aschaffenburg,POINT (9.23257 49.93527),25.827932
1,Aschaffenburg,POINT (9.2355 49.935),25.828958
2,Aschaffenburg,POINT (9.2355 49.93527),25.846390
3,Aschaffenburg,POINT (9.23591 49.935),25.713771
4,Aschaffenburg,POINT (9.23591 49.93527),25.751711


In [12]:
# Definieren Sie den Pfad zum Speichern der Datei in Google Drive
# Passen Sie den Ordner und Dateinamen bei Bedarf an
output_path_averaged_gdf = "/content/drive/MyDrive/Cold Spots Bayern/averaged_lst_per_pixel_per_city_allyears.geojson"

try:
    # Speichern Sie den GeoDataFrame als GeoJSON-Datei
    averaged_lst_per_pixel_per_city.to_file(output_path_averaged_gdf, driver='GeoJSON')
    print(f"GeoDataFrame mit gemittelten LST-Werten erfolgreich gespeichert unter: {output_path_averaged_gdf}")
except Exception as e:
    print(f"Fehler beim Speichern des GeoDataFrames: {e}")

GeoDataFrame mit gemittelten LST-Werten erfolgreich gespeichert unter: /content/drive/MyDrive/Cold Spots Bayern/averaged_lst_per_pixel_per_city_allyears.geojson


In [13]:
# Definieren Sie den Pfad, von dem die Datei geladen werden soll
# Stellen Sie sicher, dass dies mit dem Pfad übereinstimmt, unter dem Sie die Datei gespeichert haben
input_path_averaged_gdf = "/content/drive/MyDrive/Cold Spots Bayern/averaged_lst_per_pixel_per_city_allyears.geojson"

try:
    # Laden Sie den GeoDataFrame aus der GeoJSON-Datei
    loaded_averaged_lst_gdf = gpd.read_file(input_path_averaged_gdf)
    print(f"GeoDataFrame erfolgreich geladen von: {input_path_averaged_gdf}")
    print(f"Anzahl der Einträge im geladenen GeoDataFrame: {len(loaded_averaged_lst_gdf)}")
    display(loaded_averaged_lst_gdf.head())

except Exception as e:
    print(f"Fehler beim Laden des GeoDataFrames von {input_path_averaged_gdf}: {e}")
    print("Bitte überprüfen Sie den Dateipfad und stellen Sie sicher, dass die Datei existiert.")

# Nun können Sie mit 'loaded_averaged_lst_gdf' weiterarbeiten
# Zum Beispiel können Sie es in city_averaged_gdfs aufteilen oder direkt verwenden
# loaded_averaged_lst_gdf sollte die gleiche Struktur wie 'averaged_lst_per_pixel_per_city' haben

GeoDataFrame erfolgreich geladen von: /content/drive/MyDrive/Cold Spots Bayern/averaged_lst_per_pixel_per_city_allyears.geojson
Anzahl der Einträge im geladenen GeoDataFrame: 1992069


,city,avg_summer_LST_Celsius,geometry
0,Aschaffenburg,25.827932,POINT (9.23257 49.93527)
1,Aschaffenburg,25.828958,POINT (9.2355 49.935)
2,Aschaffenburg,25.846390,POINT (9.2355 49.93527)
3,Aschaffenburg,25.713771,POINT (9.23591 49.935)
4,Aschaffenburg,25.751711,POINT (9.23591 49.93527)


In [14]:
# 2. Erstellung stadtspezifischer GeoDataFrames
# Dictionary zum Speichern der stadtspezifischen GeoDataFrames mit gemittelten Werten
city_averaged_gdfs = {}

unique_cities_averaged = loaded_averaged_lst_gdf['city'].unique()
print(f"\nEindeutige Städte im gemittelten GeoDataFrame: {list(unique_cities_averaged)}")


print("\nErstelle stadtspezifische GeoDataFrames mit gemittelten LST-Werten...")

for city in unique_cities_averaged:
    print(f"  Erstelle GeoDataFrame für Stadt: {city}...")
    gdf_city_averaged = loaded_averaged_lst_gdf[
        loaded_averaged_lst_gdf['city'] == city
    ].copy() # Wichtig: Kopie erstellen

    if not gdf_city_averaged.empty:
        city_averaged_gdfs[city] = gdf_city_averaged
        print(f"    GeoDataFrame für {city} erstellt mit {len(gdf_city_averaged)} Pixeln.")
    else:
        print(f"    Keine gemittelten Daten für Stadt {city} gefunden.")

print("\nStadtspezifische GeoDataFrames mit gemittelten LST-Werten erstellt.")

# Jetzt haben Sie ein Dictionary 'city_averaged_gdfs', das für jede Stadt einen GeoDataFrame
# mit den gemittelten LST-Werten pro Pixel über alle Jahre enthält.
# Beispiel: city_averaged_gdfs['Munich'] gibt den GeoDataFrame für München mit gemittelten LST-Werten zurück.


Eindeutige Städte im gemittelten GeoDataFrame: ['Aschaffenburg', 'Augsburg', 'Bamberg', 'Bayreuth', 'Erlangen', 'Fürth', 'Ingolstadt', 'Kempten (Allgäu)', 'Landshut', 'Munich', 'Nuremberg', 'Passau', 'Regensburg', 'Rosenheim', 'Schweinfurt', 'Würzburg']

Erstelle stadtspezifische GeoDataFrames mit gemittelten LST-Werten...
  Erstelle GeoDataFrame für Stadt: Aschaffenburg...
    GeoDataFrame für Aschaffenburg erstellt mit 69391 Pixeln.
  Erstelle GeoDataFrame für Stadt: Augsburg...
    GeoDataFrame für Augsburg erstellt mit 164425 Pixeln.
  Erstelle GeoDataFrame für Stadt: Bamberg...
    GeoDataFrame für Bamberg erstellt mit 121604 Pixeln.
  Erstelle GeoDataFrame für Stadt: Bayreuth...
    GeoDataFrame für Bayreuth erstellt mit 149925 Pixeln.
  Erstelle GeoDataFrame für Stadt: Erlangen...
    GeoDataFrame für Erlangen erstellt mit 80705 Pixeln.
  Erstelle GeoDataFrame für Stadt: Fürth...
    GeoDataFrame für Fürth erstellt mit 60360 Pixeln.
  Erstelle GeoDataFrame für Stadt: Ingolstadt

In [ ]:
# 3. KNN-Matrix Berechnung pro Stadt
# Dictionary zum Speichern der KNN-Gewichtsmatrizen für jede Stadt
city_knn_weights_averaged = {}

# Definieren Sie die Anzahl der Nachbarn für KNN
k_neighbors = 12 # Sie können diesen Wert anpassen

print(f"\nBerechne KNN-Gewichtsmatrizen (k={k_neighbors}) für jede Stadt basierend auf gemittelten LST-Werten...")

# Schleife über die stadtspezifischen GeoDataFrames
for city, gdf_city_averaged in city_averaged_gdfs.items():
    print(f"\n✨ Verarbeite Stadt: {city}...")

    # Stellen Sie sicher, dass genügend Punkte für die KNN-Analyse vorhanden sind
    min_points_for_knn = k_neighbors + 1
    if not gdf_city_averaged.empty and len(gdf_city_averaged) >= min_points_for_knn:
        # Extrahieren Sie die Koordinaten
        # Konvertiere in ein metrisches CRS vor der KNN-Berechnung, wenn es noch nicht projiziert ist
        gdf_city_spatial = gdf_city_averaged.copy() # Kopie erstellen
        if gdf_city_spatial.crs is None or gdf_city_spatial.crs.is_geographic:
             # Für Bayern ist EPSG:25832 (ETRS89 / UTM zone 32N) oft passend.
             try:
                 print(f"    Konvertiere Daten für {city} nach EPSG:25832 (UTM 32N) für metrische Distanzen.")
                 gdf_city_spatial = gdf_city_spatial.to_crs(epsg=25832)
             except Exception as crs_e:
                 print(f"    Fehler bei der CRS-Konvertierung für {city}: {crs_e}. Versuche es ohne Konvertierung, Ergebnisse könnten ungenau sein.")
                 pass # Geht weiter mit den ursprünglichen Koordinaten

        coords = np.array(list(zip(gdf_city_spatial.geometry.x, gdf_city_spatial.geometry.y)))

        # Berechne die KNN-Matrix
        try:
            w_city_averaged = KNN.from_array(coords, k=k_neighbors)
            w_city_averaged.transform = 'R' # Zeilenstandardisierung anwenden

            # Prüfen auf Konnektivität
            if w_city_averaged.n_components > 1:
                 print(f"    Warnung: Gewichtsmatrix für {city} ist nicht vollständig verbunden ({w_city_averaged.n_components} Komponenten).")

            # Speichere die Gewichtsmatrix im Dictionary
            city_knn_weights_averaged[city] = w_city_averaged
            print(f"    KNN-Matrix für {city} erfolgreich berechnet.")

        except Exception as knn_e:
            print(f"    Fehler bei der KNN-Berechnung für {city}: {knn_e}. Überspringe diese Stadt.")


    else:
        print(f"  Nicht genügend Daten ({len(gdf_city_averaged)} Punkte) für KNN-Berechnung für {city}. Benötigt mindestens {min_points_for_knn} Punkte. Überspringe.")


print("\nKNN-Berechnung für alle Städte (basierend auf gemittelten LST-Werten) abgeschlossen.")

# Das Dictionary 'city_knn_weights_averaged' enthält nun die KNN-Matrizen
# für jede Stadt, basierend auf den über die Jahre gemittelten LST-Werten pro Pixel.
# Beispiel: city_knn_weights_averaged['Munich'] gibt die KNN-Matrix für München zurück.

4. **Ergebnisse speichern/anzeigen**:

Die gemittelten GeoDataFrames pro Stadt sind im Dictionary `city_averaged_gdfs` gespeichert.
Die KNN-Gewichtsmatrizen pro Stadt sind im Dictionary `city_knn_weights_averaged` gespeichert.

Sie können diese Ergebnisse nun speichern oder weiter analysieren (z.B. Gi* Statistik berechnen).

**Beispiel zum Anzeigen der ersten Zeilen des gemittelten GeoDataFrames für eine Stadt (z.B. Munich):**

In [ ]:
# 5. Gi*-Algorithmus für Cold Spot Bestimmung pro Stadt

# Dictionary zum Speichern der Ergebnisse der Gi* Analyse für jede Stadt
city_gi_results = {}

print("\nFühre Gi* Analyse (Getis-Ord Local Statistic) für jede Stadt durch...")

# Schleife über die stadtspezifischen GeoDataFrames mit gemittelten Werten
for city, gdf_city_averaged in city_averaged_gdfs.items():
    print(f"\n✨ Führe Gi* Analyse für Stadt: {city} durch...")

    # Überprüfen Sie, ob eine Gewichtsmatrix für diese Stadt verfügbar ist
    if city in city_knn_weights_averaged and city_knn_weights_averaged[city] is not None:
        w_city_averaged = city_knn_weights_averaged[city]

        # Stellen Sie sicher, dass die Gewichtsmatrix und der GeoDataFrame die gleiche Anzahl von Beobachtungen haben
        if len(gdf_city_averaged) != w_city_averaged.n:
            print(f"  ⚠️ Warnung: Anzahl der Beobachtungen im GDF ({len(gdf_city_averaged)}) und der Gewichtsmatrix ({w_city_averaged.n}) für {city} stimmen nicht überein. Überspringe Gi* Analyse für diese Stadt.")
            continue

        # Extrahieren Sie die zu analysierende Variable (gemittelte LST-Werte)
        # Stellen Sie sicher, dass die Reihenfolge der Werte im Array mit der Reihenfolge im GDF übereinstimmt.
        # Dies ist wichtig, da die Gewichtsmatrix auf der Reihenfolge der Geometrien basiert.
        y = gdf_city_averaged['avg_summer_LST_Celsius'].values

        # Optional: Standardisieren Sie die Variable, obwohl Gi* dies intern tut, ist es manchmal hilfreich
        # scaler = StandardScaler()
        # y_scaled = scaler.fit_transform(y.reshape(-1, 1)).flatten()
        # print("  Variable standardisiert.")

        # Führen Sie die Gi* Analyse durch
        try:
            # Gi* rechnet Hot Spots (hohe Werte umgeben von hohen Werten) und Cold Spots (niedrige Werte umgeben von niedrigen Werten)
            # Da wir Cold Spots suchen, interessieren uns die negativen Gi* Werte und die entsprechenden p-Werte.
            gi_local = G_Local(y, w_city_averaged)

            # Fügen Sie die Ergebnisse (Gi* Statistik und p-Werte) dem GeoDataFrame hinzu
            gdf_city_averaged[f'Gi_Star'] = gi_local.Gs
            gdf_city_averaged[f'Gi_Star_p_value'] = gi_local.p_sim # Verwenden Sie p_sim für die Signifikanz basierend auf Simulation

            # Bestimmen Sie die Signifikanz (p < 0.05)
            # Cold Spots: Negative Gi* Werte mit p-Wert < 0.05
            # Hot Spots: Positive Gi* Werte mit p-Wert < 0.05
            # Nicht signifikant: p-Wert >= 0.05
            gdf_city_averaged['Gi_Star_sig'] = 'Nicht signifikant'
            gdf_city_averaged.loc[(gdf_city_averaged['Gi_Star_p_value'] < 0.05) & (gdf_city_averaged['Gi_Star'] < 0), 'Gi_Star_sig'] = 'Cold Spot (p<0.05)'
            gdf_city_averaged.loc[(gdf_city_averaged['Gi_Star_p_value'] < 0.05) & (gdf_city_averaged['Gi_Star'] > 0), 'Gi_Star_sig'] = 'Hot Spot (p<0.05)'
            # Sie können weitere Signifikanzniveaus hinzufügen (z.B. p<0.01, p<0.001)

            # Speichere den aktualisierten GeoDataFrame mit den Gi* Ergebnissen im Dictionary
            city_gi_results[city] = gdf_city_averaged

            print(f"  Gi* Analyse für {city} erfolgreich durchgeführt.")
            # Beispiel zum Anzeigen der ersten Zeilen mit den neuen Spalten
            # display(gdf_city_averaged[['avg_summer_LST_Celsius', 'Gi_Star', 'Gi_Star_p_value', 'Gi_Star_sig']].head())


        except Exception as gi_e:
            print(f"  Fehler bei der Gi* Berechnung für {city}: {gi_e}. Überspringe diese Stadt.")

    else:
        print(f"  Keine KNN-Gewichtsmatrix für Stadt {city} verfügbar. Überspringe Gi* Analyse.")

print("\nGi* Analyse für alle Städte abgeschlossen.")

# Das Dictionary 'city_gi_results' enthält nun die GeoDataFrames für jede Stadt
# mit den hinzugefügten Spalten 'Gi_Star', 'Gi_Star_p_value' und 'Gi_Star_sig'.
# Beispiel: city_gi_results['Munich'] gibt den GeoDataFrame für München mit Gi* Ergebnissen zurück.

# Sie können nun die Ergebnisse visualisieren oder weiter analysieren.

6. **Finish task**:

Die Gi* Analyse wurde für jede Stadt basierend auf den gemittelten LST-Werten und den KNN-Gewichtsmatrizen durchgeführt. Die Ergebnisse sind in den GeoDataFrames im Dictionary `city_gi_results` enthalten. Sie beinhalten die Gi* Statistik, den p-Wert und eine Klassifizierung (Cold Spot, Hot Spot, Nicht signifikant).

Sie können nun die Cold Spots (und Hot Spots) visualisieren oder die Ergebnisse weiter analysieren.

In [ ]:
# 1. Gruppierung und Mittelwertbildung
# Gruppieren nach 'city' und 'geometry' und berechnen des Mittelwerts für 'avg_summer_LST_Celsius'
# Setzen Sie 'geometry' als Index, um die Gruppierung zu vereinfachen und dann zurückzusetzen
averaged_lst_per_pixel_per_city = all_cities_summer_avg_lst_per_pixel.set_index('geometry').groupby(['city', 'geometry'])['avg_summer_LST_Celsius'].mean().reset_index()

# Konvertieren Sie das Ergebnis zurück in ein GeoDataFrame
# Annahme: Das ursprüngliche CRS des GeoDataFrames ist bekannt oder kann von der ersten Zeile abgeleitet werden.
# Wenn das ursprüngliche CRS nicht bekannt ist, müssen Sie es hier explizit festlegen.
# Beispiel: original_crs = all_cities_summer_avg_lst_per_pixel.crs
# Wenn das CRS None ist und Sie wissen, dass es sich um WGS84 (EPSG:4326) handelt:
# original_crs = "EPSG:4326"

# Verwenden Sie das CRS des ursprünglichen GeoDataFrames
original_crs = all_cities_summer_avg_lst_per_pixel.crs

averaged_lst_per_pixel_per_city = gpd.GeoDataFrame(
    averaged_lst_per_pixel_per_city,
    geometry='geometry',
    crs=original_crs # Setzen Sie das CRS des ursprünglichen GeoDataFrames
)

print("Mittelwert der LST pro Pixel pro Stadt über alle Jahre berechnet.")
display(averaged_lst_per_pixel_per_city.head())

In [ ]:
# 2. Erstellung stadtspezifischer GeoDataFrames
# Dictionary zum Speichern der stadtspezifischen GeoDataFrames mit gemittelten Werten
city_averaged_gdfs = {}

unique_cities_averaged = averaged_lst_per_pixel_per_city['city'].unique()
print(f"\nEindeutige Städte im gemittelten GeoDataFrame: {list(unique_cities_averaged)}")


print("\nErstelle stadtspezifische GeoDataFrames mit gemittelten LST-Werten...")

for city in unique_cities_averaged:
    print(f"  Erstelle GeoDataFrame für Stadt: {city}...")
    gdf_city_averaged = averaged_lst_per_pixel_per_city[
        averaged_lst_per_pixel_per_city['city'] == city
    ].copy() # Wichtig: Kopie erstellen

    if not gdf_city_averaged.empty:
        city_averaged_gdfs[city] = gdf_city_averaged
        print(f"    GeoDataFrame für {city} erstellt mit {len(gdf_city_averaged)} Pixeln.")
    else:
        print(f"    Keine gemittelten Daten für Stadt {city} gefunden.")

print("\nStadtspezifische GeoDataFrames mit gemittelten LST-Werten erstellt.")

# Jetzt haben Sie ein Dictionary 'city_averaged_gdfs', das für jede Stadt einen GeoDataFrame
# mit den gemittelten LST-Werten pro Pixel über alle Jahre enthält.
# Beispiel: city_averaged_gdfs['Munich'] gibt den GeoDataFrame für München mit gemittelten LST-Werten zurück.

In [ ]:
# 3. KNN-Matrix Berechnung pro Stadt
# Dictionary zum Speichern der KNN-Gewichtsmatrizen für jede Stadt
city_knn_weights_averaged = {}

# Definieren Sie die Anzahl der Nachbarn für KNN
k_neighbors = 12 # Sie können diesen Wert anpassen

print(f"\nBerechne KNN-Gewichtsmatrizen (k={k_neighbors}) für jede Stadt basierend auf gemittelten LST-Werten...")

# Schleife über die stadtspezifischen GeoDataFrames
for city, gdf_city_averaged in city_averaged_gdfs.items():
    print(f"\n✨ Verarbeite Stadt: {city}...")

    # Stellen Sie sicher, dass genügend Punkte für die KNN-Analyse vorhanden sind
    min_points_for_knn = k_neighbors + 1
    if not gdf_city_averaged.empty and len(gdf_city_averaged) >= min_points_for_knn:
        # Extrahieren Sie die Koordinaten
        # Konvertiere in ein metrisches CRS vor der KNN-Berechnung, wenn es noch nicht projiziert ist
        gdf_city_spatial = gdf_city_averaged.copy() # Kopie erstellen
        if gdf_city_spatial.crs is None or gdf_city_spatial.crs.is_geographic:
             # Für Bayern ist EPSG:25832 (ETRS89 / UTM zone 32N) oft passend.
             try:
                 print(f"    Konvertiere Daten für {city} nach EPSG:25832 (UTM 32N) für metrische Distanzen.")
                 gdf_city_spatial = gdf_city_spatial.to_crs(epsg=25832)
             except Exception as crs_e:
                 print(f"    Fehler bei der CRS-Konvertierung für {city}: {crs_e}. Versuche es ohne Konvertierung, Ergebnisse könnten ungenau sein.")
                 pass # Geht weiter mit den ursprünglichen Koordinaten

        coords = np.array(list(zip(gdf_city_spatial.geometry.x, gdf_city_spatial.geometry.y)))

        # Berechne die KNN-Matrix
        try:
            w_city_averaged = KNN.from_array(coords, k=k_neighbors)
            w_city_averaged.transform = 'R' # Zeilenstandardisierung anwenden

            # Prüfen auf Konnektivität
            if w_city_averaged.n_components > 1:
                 print(f"    Warnung: Gewichtsmatrix für {city} ist nicht vollständig verbunden ({w_city_averaged.n_components} Komponenten).")

            # Speichere die Gewichtsmatrix im Dictionary
            city_knn_weights_averaged[city] = w_city_averaged
            print(f"    KNN-Matrix für {city} erfolgreich berechnet.")

        except Exception as knn_e:
            print(f"    Fehler bei der KNN-Berechnung für {city}: {knn_e}. Überspringe diese Stadt.")


    else:
        print(f"  Nicht genügend Daten ({len(gdf_city_averaged)} Punkte) für KNN-Berechnung für {city}. Benötigt mindestens {min_points_for_knn} Punkte. Überspringe.")


print("\nKNN-Berechnung für alle Städte (basierend auf gemittelten LST-Werten) abgeschlossen.")

# Das Dictionary 'city_knn_weights_averaged' enthält nun die KNN-Matrizen
# für jede Stadt, basierend auf den über die Jahre gemittelten LST-Werten pro Pixel.
# Beispiel: city_knn_weights_averaged['Munich'] gibt die KNN-Matrix für München zurück.

4. **Ergebnisse speichern/anzeigen**:

Die gemittelten GeoDataFrames pro Stadt sind im Dictionary `city_averaged_gdfs` gespeichert.
Die KNN-Gewichtsmatrizen pro Stadt sind im Dictionary `city_knn_weights_averaged` gespeichert.

Sie können diese Ergebnisse nun speichern oder weiter analysieren (z.B. Gi* Statistik berechnen).

**Beispiel zum Anzeigen der ersten Zeilen des gemittelten GeoDataFrames für eine Stadt (z.B. Munich):**

# Gi*-Algorithmus für Cold Spot Bestimmung

In [ ]:
# 5. Gi*-Algorithmus für Cold Spot Bestimmung pro Stadt

# Dictionary zum Speichern der Ergebnisse der Gi* Analyse für jede Stadt
city_gi_results = {}

print("\nFühre Gi* Analyse (Getis-Ord Local Statistic) für jede Stadt durch...")

# Schleife über die stadtspezifischen GeoDataFrames mit gemittelten Werten
for city, gdf_city_averaged in city_averaged_gdfs.items():
    print(f"\n✨ Führe Gi* Analyse für Stadt: {city} durch...")

    # Überprüfen Sie, ob eine Gewichtsmatrix für diese Stadt verfügbar ist
    if city in city_knn_weights_averaged and city_knn_weights_averaged[city] is not None:
        w_city_averaged = city_knn_weights_averaged[city]

        # Stellen Sie sicher, dass die Gewichtsmatrix und der GeoDataFrame die gleiche Anzahl von Beobachtungen haben
        if len(gdf_city_averaged) != w_city_averaged.n:
            print(f"  ⚠️ Warnung: Anzahl der Beobachtungen im GDF ({len(gdf_city_averaged)}) und der Gewichtsmatrix ({w_city_averaged.n}) für {city} stimmen nicht überein. Überspringe Gi* Analyse für diese Stadt.")
            continue

        # Extrahieren Sie die zu analysierende Variable (gemittelte LST-Werte)
        # Stellen Sie sicher, dass die Reihenfolge der Werte im Array mit der Reihenfolge im GDF übereinstimmt.
        # Dies ist wichtig, da die Gewichtsmatrix auf der Reihenfolge der Geometrien basiert.
        y = gdf_city_averaged['avg_summer_LST_Celsius'].values

        # Optional: Standardisieren Sie die Variable, obwohl Gi* dies intern tut, ist es manchmal hilfreich
        # scaler = StandardScaler()
        # y_scaled = scaler.fit_transform(y.reshape(-1, 1)).flatten()
        # print("  Variable standardisiert.")

        # Führen Sie die Gi* Analyse durch
        try:
            # Gi* rechnet Hot Spots (hohe Werte umgeben von hohen Werten) und Cold Spots (niedrige Werte umgeben von niedrigen Werten)
            # Da wir Cold Spots suchen, interessieren uns die negativen Gi* Werte und die entsprechenden p-Werte.
            gi_local = G_Local(y, w_city_averaged)

            # Fügen Sie die Ergebnisse (Gi* Statistik und p-Werte) dem GeoDataFrame hinzu
            gdf_city_averaged[f'Gi_Star'] = gi_local.Gs
            gdf_city_averaged[f'Gi_Star_p_value'] = gi_local.p_sim # Verwenden Sie p_sim für die Signifikanz basierend auf Simulation

            # Bestimmen Sie die Signifikanz (p < 0.05)
            # Cold Spots: Negative Gi* Werte mit p-Wert < 0.05
            # Hot Spots: Positive Gi* Werte mit p-Wert < 0.05
            # Nicht signifikant: p-Wert >= 0.05
            gdf_city_averaged['Gi_Star_sig'] = 'Nicht signifikant'
            gdf_city_averaged.loc[(gdf_city_averaged['Gi_Star_p_value'] < 0.05) & (gdf_city_averaged['Gi_Star'] < 0), 'Gi_Star_sig'] = 'Cold Spot (p<0.05)'
            gdf_city_averaged.loc[(gdf_city_averaged['Gi_Star_p_value'] < 0.05) & (gdf_city_averaged['Gi_Star'] > 0), 'Gi_Star_sig'] = 'Hot Spot (p<0.05)'
            # Sie können weitere Signifikanzniveaus hinzufügen (z.B. p<0.01, p<0.001)

            # Speichere den aktualisierten GeoDataFrame mit den Gi* Ergebnissen im Dictionary
            city_gi_results[city] = gdf_city_averaged

            print(f"  Gi* Analyse für {city} erfolgreich durchgeführt.")
            # Beispiel zum Anzeigen der ersten Zeilen mit den neuen Spalten
            # display(gdf_city_averaged[['avg_summer_LST_Celsius', 'Gi_Star', 'Gi_Star_p_value', 'Gi_Star_sig']].head())


        except Exception as gi_e:
            print(f"  Fehler bei der Gi* Berechnung für {city}: {gi_e}. Überspringe diese Stadt.")

    else:
        print(f"  Keine KNN-Gewichtsmatrix für Stadt {city} verfügbar. Überspringe Gi* Analyse.")

print("\nGi* Analyse für alle Städte abgeschlossen.")

# Das Dictionary 'city_gi_results' enthält nun die GeoDataFrames für jede Stadt
# mit den hinzugefügten Spalten 'Gi_Star', 'Gi_Star_p_value' und 'Gi_Star_sig'.
# Beispiel: city_gi_results['Munich'] gibt den GeoDataFrame für München mit Gi* Ergebnissen zurück.

# Sie können nun die Ergebnisse visualisieren oder weiter analysieren.